# CV552 Sign-Language Recognition — Evaluation & Architecture Comparison

Consumes the artifacts produced by `01_training_pipelines.ipynb` (models, SVM features, training histories) from Google Drive and computes the comparative metrics required by the project rubric.

## 1. Mount Drive & locate artifacts

In [1]:
# ==========================================
# Mount Drive + locate artifacts written by 01_training_pipelines.ipynb
# ==========================================
import os, json, glob
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT   = '/content/drive/MyDrive/CV552_SignLanguage'
MODELS_DIR   = os.path.join(DRIVE_ROOT, 'models')
FEATURES_DIR = os.path.join(DRIVE_ROOT, 'features')
HISTORY_DIR  = os.path.join(DRIVE_ROOT, 'history')

print('Models   :', sorted(os.listdir(MODELS_DIR)))
print('Features :', sorted(os.listdir(FEATURES_DIR)))
print('History  :', sorted(os.listdir(HISTORY_DIR)))


ModuleNotFoundError: No module named 'google.colab'

## 2. Reload the Sign-MNIST test split

In [ ]:
# ==========================================
# Reload the Sign-MNIST test split (identical preprocessing as training notebook)
# ==========================================
import os, numpy as np, pandas as pd
os.environ['KAGGLE_USERNAME'] = "abdullahashiry"
os.environ['KAGGLE_KEY']      = "KGAT_331632d901a6cb7a05431b55135bd8c2"
!kaggle datasets download -d datamunge/sign-language-mnist --unzip -q

test_path = 'sign_mnist_test/sign_mnist_test.csv'
if not os.path.exists(test_path):
    test_path = 'sign_mnist_test.csv'
test_df = pd.read_csv(test_path)

y_test = test_df['label'].values
x_test = test_df.drop('label', axis=1).values.reshape(-1, 28, 28)

alphabet_mapping = {0:'A',1:'B',2:'C',3:'D',4:'E',5:'F',6:'G',7:'H',8:'I',
                    10:'K',11:'L',12:'M',13:'N',14:'O',15:'P',16:'Q',17:'R',
                    18:'S',19:'T',20:'U',21:'V',22:'W',23:'X',24:'Y'}
target_names = [alphabet_mapping[i] for i in sorted(set(y_test))]
print('Test set:', x_test.shape, y_test.shape)


## 3. Load every trained model

In [ ]:
import joblib, numpy as np, tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_preprocess

svm_model       = joblib.load(os.path.join(MODELS_DIR, 'svm_hog.joblib'))
svm_feats       = np.load(os.path.join(FEATURES_DIR, 'svm_hog_features.npz'))
x_test_features = svm_feats['x_test']

def fix_lambda_layers(model, preprocess_fn):
    """Patch Lambda layers by position: 1st=grayscale→RGB, 2nd=preprocess_input."""
    lambdas = [l for l in model.layers if type(l).__name__ == 'Lambda']
    if len(lambdas) >= 1:
        lambdas[0].function = tf.image.grayscale_to_rgb
    if len(lambdas) >= 2:
        lambdas[1].function = preprocess_fn

def load_keras(name, custom_objects=None, preprocess_fn=None):
    path  = os.path.join(MODELS_DIR, name)
    model = tf.keras.models.load_model(path, compile=False, safe_mode=False,
                                       custom_objects=custom_objects)
    if preprocess_fn is not None:
        fix_lambda_layers(model, preprocess_fn)
    return model

cnn_custom    = load_keras('custom_cnn.keras')
cnn_augmented = load_keras('cnn_augmented.keras')
mobilenet     = load_keras('mobilenetv2_head.keras',
                           custom_objects={'preprocess_input': mob_preprocess},
                           preprocess_fn=mob_preprocess)
effnet_head   = load_keras('efficientnetb0_head.keras',
                           custom_objects={'preprocess_input': eff_preprocess},
                           preprocess_fn=eff_preprocess)
effnet_ft     = load_keras('efficientnetb0_finetuned.keras',
                           custom_objects={'preprocess_input': eff_preprocess},
                           preprocess_fn=eff_preprocess)

print('All models loaded.')

## 4. Comparative metrics table

In [ ]:
# ==========================================
# Comparative metrics: accuracy, precision, recall, F1, confusion matrices
# ==========================================
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
import matplotlib.pyplot as plt
import seaborn as sns

x_test_dl = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test_eff = x_test.reshape(-1, 28, 28, 1).astype('float32')  # EfficientNet preprocess is baked in

predictions = {
    'SVM + HOG':                   svm_model.predict(x_test_features),
    'Custom CNN':                  cnn_custom.predict(x_test_dl).argmax(axis=1),
    'CNN + Augmentation':          cnn_augmented.predict(x_test_dl).argmax(axis=1),
    'MobileNetV2 (head)':          mobilenet.predict(x_test_dl).argmax(axis=1),
    'EfficientNetB0 (head)':       effnet_head.predict(x_test_eff).argmax(axis=1),
    'EfficientNetB0 (fine-tuned)': effnet_ft.predict(x_test_eff).argmax(axis=1),
}

rows = []
for name, y_pred in predictions.items():
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    rows.append({'model': name,
                 'accuracy':  accuracy_score(y_test, y_pred),
                 'precision': p, 'recall': r, 'f1_macro': f})

results_df = pd.DataFrame(rows).sort_values('accuracy', ascending=False)
print(results_df.to_string(index=False))


## 5. Confusion matrices & training curves

In [ ]:
# ==========================================
# Per-model confusion matrices and training curves
# ==========================================
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.show()

# Plot saved training curves
import glob
for hist_path in sorted(glob.glob(os.path.join(HISTORY_DIR, '*.json'))):
    with open(hist_path) as f:
        h = json.load(f)
    name = os.path.basename(hist_path).replace('.json', '')
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(h.get('accuracy', []),     label='train')
    plt.plot(h.get('val_accuracy', []), label='val')
    plt.title(f'{name} - accuracy'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(h.get('loss', []),     label='train')
    plt.plot(h.get('val_loss', []), label='val')
    plt.title(f'{name} - loss'); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
